In [0]:
import shutil
import time
import random
from pathlib import Path

In [0]:
SOURCE_VOLUME = Path("/Volumes/pipeline_1/bronze/source_volume")
TARGET_VOLUME = Path("/Volumes/pipeline_1/bronze/landing_volume")

MIN_DELAY_SEC = 1
MAX_DELAY_SEC = 5

FILE_PATTERN = "*.json"

In [0]:
def get_files_to_copy():
    source_files = sorted(SOURCE_VOLUME.glob(FILE_PATTERN))
    target_files = {f.name for f in TARGET_VOLUME.glob(FILE_PATTERN)}

    # только те, которых ещё нет
    return [f for f in source_files if f.name not in target_files]


def copy_atomic(src: Path, dst: Path):
    """
    Atomic copy:
    сначала копируем во временный файл,
    потом rename → Auto Loader не читает half-written files
    """
    temp_dst = dst.with_suffix(".tmp")

    shutil.copy2(src, temp_dst)
    temp_dst.rename(dst)


In [0]:
def run():
    print("Starting file arrival simulator...")
    print(f"Source: {SOURCE_VOLUME}")
    print(f"Target: {TARGET_VOLUME}")

    TARGET_VOLUME.mkdir(parents=True, exist_ok=True)

    while True:
        files = get_files_to_copy()

        if not files:
            print("No more files to copy. Waiting...")
            time.sleep(5)
            continue

        file_to_copy = files[0]

        destination = TARGET_VOLUME / file_to_copy.name

        copy_atomic(file_to_copy, destination)

        print(f"Copied → {file_to_copy.name}")

        delay = random.randint(MIN_DELAY_SEC, MAX_DELAY_SEC)
        time.sleep(delay)


In [0]:
run()